In [ ]:
import os
import sys
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import importlib
import portfolio.ranking
importlib.reload(portfolio.ranking)
import portfolio.validation
importlib.reload(portfolio.validation)

from portfolio.validation import run_backtest, compute_validation_stats
from IPython.display import display, HTML

# --- Run backtest ---
results = run_backtest(lookback_months=12)
stats = compute_validation_stats(results)

# --- Build HTML report ---
html = []

# CSS
html.append('<style>')
html.append('.val-wrap { font-family: Arial, sans-serif; max-width: 1400px; }')
html.append('.val-title { background: #2C3E50; color: white; padding: 14px 20px; font-size: 18px; font-weight: bold; border-radius: 6px 6px 0 0; }')
html.append('.val-sub { background: #34495E; color: #BDC3C7; padding: 8px 20px; font-size: 12px; }')
html.append('.val-table { width: 100%; border-collapse: collapse; font-size: 12px; }')
html.append('.val-table th { background: #2C3E50; color: white; padding: 8px 10px; text-align: left; position: sticky; top: 0; }')
html.append('.val-table td { padding: 6px 10px; border-bottom: 1px solid #e0e0e0; color: #1a1a1a; }')
html.append('.val-table tr:hover { filter: brightness(0.95); }')
html.append('.stat-card { display: inline-block; background: #f8f9fa; border: 1px solid #dee2e6; border-radius: 8px; padding: 12px 20px; margin: 6px; text-align: center; min-width: 140px; }')
html.append('.stat-val { font-size: 24px; font-weight: bold; }')
html.append('.stat-label { font-size: 11px; color: #666; margin-top: 4px; }')
html.append('.q-table { border-collapse: collapse; font-size: 13px; margin: 10px 0; }')
html.append('.q-table th { background: #2C3E50; color: white; padding: 8px 14px; }')
html.append('.q-table td { padding: 6px 14px; border-bottom: 1px solid #e0e0e0; }')
html.append('</style>')

html.append('<div class="val-wrap">')
html.append('<div class="val-title">Ranking Model Backtest Validation</div>')
html.append(f'<div class="val-sub">Simulated ranking from 12 months ago vs actual returns | {stats.get("total_stocks", 0)} stocks analyzed</div>')

# --- Summary cards ---
html.append('<div style="padding: 16px 20px;">')

if "error" not in stats:
    # Spearman correlation
    corr = stats["spearman_correlation"]
    corr_color = "#1B5E20" if corr > 0.3 else "#E65100" if corr > 0 else "#B71C1C"
    html.append(f'<div class="stat-card"><div class="stat-val" style="color:{corr_color};">{corr:.3f}</div><div class="stat-label">Spearman Correlation<br>(score vs return)</div></div>')

    # Hit rate
    hr = stats["hit_rate"]
    hr_color = "#1B5E20" if hr > 60 else "#E65100" if hr > 50 else "#B71C1C"
    html.append(f'<div class="stat-card"><div class="stat-val" style="color:{hr_color};">{hr:.0f}%</div><div class="stat-label">Hit Rate<br>(top half beats median)</div></div>')

    # Win rate
    wr = stats["win_rate_top10"]
    wr_color = "#1B5E20" if wr > 70 else "#E65100" if wr > 50 else "#B71C1C"
    html.append(f'<div class="stat-card"><div class="stat-val" style="color:{wr_color};">{wr:.0f}%</div><div class="stat-label">Win Rate<br>(top 10 positive return)</div></div>')

    # Top 5 avg return
    t5 = stats["top5_avg_return"]
    t5_color = "#1B5E20" if t5 > 0 else "#B71C1C"
    html.append(f'<div class="stat-card"><div class="stat-val" style="color:{t5_color};">{t5:+.1f}%</div><div class="stat-label">Top 5 Avg Return</div></div>')

    # Top 10 avg return
    t10 = stats["top10_avg_return"]
    t10_color = "#1B5E20" if t10 > 0 else "#B71C1C"
    html.append(f'<div class="stat-card"><div class="stat-val" style="color:{t10_color};">{t10:+.1f}%</div><div class="stat-label">Top 10 Avg Return</div></div>')

    # Bottom 10 avg return
    b10 = stats["bottom10_avg_return"]
    b10_color = "#1B5E20" if b10 > 0 else "#B71C1C"
    html.append(f'<div class="stat-card"><div class="stat-val" style="color:{b10_color};">{b10:+.1f}%</div><div class="stat-label">Bottom 10 Avg Return</div></div>')

html.append('</div>')

# --- Quintile table ---
if "error" not in stats:
    html.append('<div style="padding: 0 20px;">')
    html.append('<h3 style="color:#2C3E50; margin-bottom:8px;">Quintile Analysis</h3>')
    html.append('<table class="q-table">')
    html.append('<tr><th>Quintile</th><th>Avg Score</th><th>Avg Return</th><th>Spread vs Bottom</th><th>Stocks</th></tr>')
    q_data = stats["quintiles"]
    bottom_ret = q_data["Q5"]["avg_return"]
    for qk in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
        q = q_data[qk]
        spread = q["avg_return"] - bottom_ret
        ret_color = "#1B5E20" if q["avg_return"] > 0 else "#B71C1C"
        sp_color = "#1B5E20" if spread > 0 else "#B71C1C" if spread < 0 else "#666"
        tickers_str = ", ".join(q["stocks"][:6])
        if len(q["stocks"]) > 6:
            tickers_str += f" +{len(q['stocks'])-6}"
        html.append(f'<tr>')
        html.append(f'<td><b>{q["label"]}</b></td>')
        html.append(f'<td>{q["avg_score"]:.0f}</td>')
        html.append(f'<td style="color:{ret_color}; font-weight:bold;">{q["avg_return"]:+.1f}%</td>')
        html.append(f'<td style="color:{sp_color};">{spread:+.1f}%</td>')
        html.append(f'<td style="font-size:11px;">{tickers_str}</td>')
        html.append(f'</tr>')
    html.append('</table>')
    html.append('</div>')

# --- Full results table ---
html.append('<div style="padding: 16px 20px;">')
html.append('<h3 style="color:#2C3E50; margin-bottom:8px;">Full Backtest Results</h3>')
html.append('<div style="max-height: 800px; overflow-y: auto;">')
html.append('<table class="val-table">')
html.append('<tr>')
html.append('<th>#</th><th>Ticker</th><th>What</th><th>Basket</th><th>Strategy</th>')
html.append('<th>Price Then</th><th>Price Now</th><th>Return</th><th>Score</th>')
html.append('<th>Upside</th><th>Growth</th><th>Val.</th><th>LT</th><th>Cash</th><th>Conv.</th><th>Entry</th><th>Mom.</th>')
html.append('<th>Profit</th><th>Frag.</th><th>Down.</th>')
html.append('</tr>')

# Basket colors
def basket_bg(basket):
    colors = {
        "Core ETF": "#E3F2FD",
        "Nuclear": "#FFF8DC",
        "Quantum": "#F3E6F5",
        "Cyber": "#FFEBEE",
        "Industrial": "#E8EAF6",
        "SpecGrowth": "#E0F7FA",
        "MedTech": "#E8F5E9",
        "Defense": "#FFF3E0",
    }
    return colors.get(basket, "#FFFFFF")

# Strategy badges
def strategy_badge(s):
    colors = {"hold_forever": "#1B5E20", "cycle": "#E65100", "catalyst": "#B71C1C"}
    labels = {"hold_forever": "HOLD FOREVER", "cycle": "CYCLE", "catalyst": "CATALYST"}
    c = colors.get(s, "#666")
    l = labels.get(s, s.upper())
    return f'<span style="background:{c}; color:white; padding:2px 6px; border-radius:3px; font-size:10px;">{l}</span>'

for r in results:
    bg = basket_bg(r["basket"])
    bd = r.get("breakdown", {})

    ret = r.get("actual_return_pct")
    if ret is not None:
        ret_color = "#1B5E20" if ret > 0 else "#B71C1C"
        ret_str = f'{ret:+.1f}%'
    else:
        ret_color = "#999"
        ret_str = "—"

    # Score color
    sc = r["score"]
    if sc >= 60:
        sc_color = "#1B5E20"
    elif sc >= 40:
        sc_color = "#E65100"
    else:
        sc_color = "#B71C1C"

    html.append(f'<tr style="background:{bg};">')
    html.append(f'<td><b>{r["rank"]}</b></td>')
    html.append(f'<td><a href="https://finance.yahoo.com/quote/{r["ticker"]}" target="_blank" style="color:#1565C0; text-decoration:none;">{r["ticker"]}</a></td>')
    html.append(f'<td style="max-width:200px; font-size:11px;">{r["what"]}</td>')
    html.append(f'<td>{r["basket"]}</td>')
    html.append(f'<td>{strategy_badge(r["strategy"])}</td>')
    html.append(f'<td>${r["price_then"]:,.2f}</td>')
    html.append(f'<td>${r["price_now"]:,.2f}</td>' if r["price_now"] else '<td>—</td>')
    html.append(f'<td style="color:{ret_color}; font-weight:bold;">{ret_str}</td>')
    html.append(f'<td style="color:{sc_color}; font-weight:bold;">{sc:.0f}</td>')

    # Breakdown sub-scores
    for key in ["upside", "growth", "valuation", "long_term", "cash_runway", "conviction", "entry", "momentum"]:
        html.append(f'<td style="text-align:center; font-size:10px;">{bd.get(key, 0):.0f}</td>')

    # Risk adjustments
    for key in ["profitability", "fragility", "downside"]:
        val = bd.get(key, 0)
        c = "#1B5E20" if val > 0 else "#B71C1C" if val < 0 else "#999"
        html.append(f'<td style="text-align:center; font-size:10px; color:{c};">{val:+.0f}</td>')

    html.append('</tr>')

html.append('</table>')
html.append('</div>')
html.append('</div>')

# --- Legend ---
html.append('<div style="padding: 16px 20px; font-size: 12px; color: #555; border-top: 1px solid #e0e0e0;">')
html.append('<b>How to read this:</b>')
html.append('<br>• <b>Spearman Correlation:</b> Measures if higher-scored stocks had higher returns. +1.0 = perfect, 0 = random, -1.0 = inverse. Above +0.3 is good.')
html.append('<br>• <b>Hit Rate:</b> % of top-half ranked stocks that beat the median return. Above 60% means the model separates winners from losers.')
html.append('<br>• <b>Win Rate:</b> % of top 10 stocks with positive returns. Above 70% is strong.')
html.append('<br>• <b>Quintile Spread:</b> If Q1 (top 20%) significantly outperforms Q5 (bottom 20%), the model has predictive power.')
html.append('<br>• <b>Caveat:</b> Analyst targets and consensus use current values (not available historically). Revenue growth is reconstructed from quarterly filings.')
html.append('</div>')

html.append('</div>')

display(HTML('\n'.join(html)))
